In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
select * from workspace.silver.silver_flights

###Parameters

In [0]:
#KEY columns
dbutils.widgets.text("keycols", "['flight_id']")


# CDC COLUMN
dbutils.widgets.text("cdccol", "")

# BACKDATED REFRESH
dbutils.widgets.text("backdated_refresh", "")


# source Object
dbutils.widgets.text("source_object", "")

# source schema
dbutils.widgets.text("source_schema", "")

### **Fetching Parameters & Creating Variables**


In [0]:
# # Catalog Name
# catalog = "workspace"

# # Key Cols List
# key_cols = "['flight_id']"
# key_cols_list = eval(key_cols)

# # CDC Column
# cdc_col = "modifiedDate"

# # Backdated Refresh
# backdated_refresh = ""

# # Source Object
# source_object = "silver_flights"

# # Source Schema
# source_schema = "silver"

# # target schema
# target_schema = "gold"

# # target object
# target_object = "DimFlights" 

# # surrogate key column
# surrogate_key = "DimFlightsKey"

In [0]:
# Catalog Name
catalog = "workspace"

# Key Cols List
key_cols = "['airport_id']"
key_cols_list = eval(key_cols)

# CDC Column
cdc_col = "modifiedDate"

# Backdated Refresh
backdated_refresh = ""

# Source Object
source_object = "silver_airports"

# Source Schema
source_schema = "silver"

# target schema
target_schema = "gold"

# target object
target_object = "DimAirports" 

# surrogate key column
surrogate_key = "DimAirportsKey"

In [0]:
# # Catalog Name
# catalog = "workspace"

# # Key Cols List
# key_cols = "['passenger_id']"
# key_cols_list = eval(key_cols)

# # CDC Column
# cdc_col = "modifiedDate"

# # Backdated Refresh
# backdated_refresh = ""

# # Source Object
# source_object = "silver_passengers"

# # Source Schema
# source_schema = "silver"

# # target schema
# target_schema = "gold"

# # target object
# target_object = "DimPassengers" 

# # surrogate key column
# surrogate_key = "DimPassengersKey"

### **INCREMENTAL DATA INGESTION**

In [0]:
if len(backdated_refresh) == 0:

    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    
        last_load = spark.sql(f"SELECT max({cdc_col}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
    
    else:
        last_load = "1900-01-01 00:00:00"
        
# yes back date refresh
else:
    last_load = backdated_refresh



In [0]:
df_src = spark.sql(f"SELECT * FROM workspace.{source_schema}.{source_object} WHERE {cdc_col} >= TIMESTAMP '{last_load}'")

#### OLD vs NEW RECORDS

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    # Key Columns String For Incremental
    key_cols_string_incremental = ", ".join(key_cols_list)
    df_trg = spark.sql(
        f"""
        SELECT {key_cols_string_incremental}, {surrogate_key}, create_date, update_date
        FROM {catalog}.{target_schema}.{target_object}
        """
    )
else:
    # Key Columns String For Initial
    key_cols_string_init = ", ".join([f"'' AS {col}" for col in key_cols_list])
    df_trg = spark.sql(
        f"""
        SELECT {key_cols_string_init}, CAST('0' AS INT) AS {surrogate_key}, CAST ('1900-01-01 00:00:00' as timestamp) AS create_date, CAST ('1900-01-01 00:00:00' as timestamp) AS update_date
        """
    )

In [0]:
df_trg.display()

### JOIN CONDITION

In [0]:
join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_cols_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

df_join = spark.sql(
    f"""
          SELECT src.*,
          trg.{surrogate_key},
          trg.create_date,
          trg.update_date
          FROM src
          LEFT JOIN trg
          ON {join_condition}
          """
)

In [0]:
df_join.display()

In [0]:
# OLD Records
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
# new records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())


In [0]:
df_old.display()

### ENRICHING DFS

###Preparing DF_OLD###

In [0]:
df_old_enr = df_old.withColumn("update_date", current_timestamp())


###Preparing DF_New###


In [0]:
df_new.display()

In [0]:
if spark.catalog.tableExists(f"""{catalog}.{target_schema}.{target_object}"""):
    max_surrogate_key = spark.sql(f"SELECT MAX({surrogate_key}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
    df_new_enr = df_new.withColumn(f"{surrogate_key}", lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())\
        .withColumn("create_date", current_timestamp())\
        .withColumn("update_date", current_timestamp())
else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f"{surrogate_key}", lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())\
        .withColumn("create_date", current_timestamp())\
        .withColumn("update_date", current_timestamp())



In [0]:
df_new_enr.display()

In [0]:
df_old_enr.display()

#### **Unioning OLD AND NEW RECORDS**####

In [0]:
df_union = df_old_enr.unionByName(df_new_enr)

In [0]:
df_union.display()

### UPSERT

In [0]:
df_union.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
        .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
        .whenNotMatchedInsertAll()\
        .execute()
      
else:
    df_union.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

In [0]:
%sql
select * from workspace.gold.dimflights

#### **GOLD_FACT**

In [0]:
# Catalog Name
catalog = "workspace"

# Source Schema
source_schema = "silver"

# Source Object
source_object = "silver_bookings"

# CDC Column
cdc_column = "modifiedDate"

# Backdated Refresh
backdated_refresh = ""

# Source Fact Table
fact_table = f"{catalog}.{source_schema}.{source_object}"
# target schema
target_schema = "gold"

# target object
target_object = "FactBookings" 

# Fact Key Cols List
fact_key_cols = ["DimPassengersKey", "DimFlightsKey", "DimAirportsKey",  "booking_date"]

In [0]:
dimensions = [
    {
        "table": f"{catalog}.{target_schema}.DimPassengers",
        "alias": "DimPassengers",
        "join_keys": [("passenger_id", "passenger_id")], # (fact_col, dim_Col)
    },
    {
        "table": f"{catalog}.{target_schema}.DimFlights",
        "alias": "DimFlights",
        "join_keys": [("flight_id", "flight_id")], # (fact_col, dim_Col)
    },
    {
        "table": f"{catalog}.{target_schema}.DimAirports",
        "alias": "DimAirports",
        "join_keys": [("airport_id", "airport_id")], # (fact_col, dim_Col)
    }
]

# Columns you want to keep from Face table (besides the surrogate keys)
fact_columns = ["amount", "booking_date", "modifiedDate"]

In [0]:
if len(backdated_refresh) == 0:

    if spark.catalog.tableExists(f"{catalog}.{source_schema}.{target_object}"):
    
        last_load = spark.sql(
            f"""SELECT max({cdc_column}) FROM workspace.{target_schema}.{target_object}"""
        ).collect()[0][0]
    
    else:
    
        last_load = "1900-01-01 00:00:00"

else:

    last_load = backdated_refresh

last_load

In [0]:
def generate_fact_query_incremental(
    fact_table, dimensions, fact_columns, cdc_column, processing_date
):
    fact_alias = "f"

    # Base columns to select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # Build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        table_name = table_full.split(".")[-1]
        surrogate_key = f"{alias}.{table_name}Key"
        select_cols.append(surrogate_key)

        # Build On clause
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{fk}" for fk, dk in dim["join_keys"]
        ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(
            on_conditions
        )
        join_clauses.append(join_clause)
    # Final SELECT and join clauses
    select_clause = ",\n    ".join(select_cols)
    joins = "\n".join(join_clauses)

    # WHERE clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_column} >= DATE('{last_load}')"

    # Final query
    query = f"""
        SELECT 
        {select_clause}
        FROM
        {fact_table} {fact_alias}
        {joins}
        WHERE
        {where_clause}
    """.strip()

    return query

In [0]:
query = generate_fact_query_incremental(
    fact_table, dimensions, fact_columns, cdc_column, last_load
)

In [0]:
print(query)

In [0]:
df_fact = spark.sql(query)

In [0]:
# Fact Key COlumns Merge Condition
fact_key_cols

In [0]:
fact_key_cols_str = " AND ".join([f"src.{col}" for col in fact_key_cols])

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
            .whenMatchedUpdateAll(condition = f"src.{cdc_column} >= trg.{cdc_column}")\
            .whenNotMatchedInsertAll()\
            .execute()

else:

    df_fact.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

In [0]:
%sql
select * from workspace.gold.factbookings